[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 07](README.md)

# Híbrido MPI + GPU

**Tema:** 07 · **Sesiones:** 32, 34 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cómo asignar procesos a dispositivos y solapar halos sin ocultar transferencias?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** Combinar MPI y GPU añade una asignación rank–dispositivo y rutas alternativas para halos. La topología física pasa a ser parte del algoritmo.

**Prerrequisitos.**

- MPI, OpenMP y un modelo de acelerador.
- Afinidad, escalabilidad y lectura de perfiles.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Construir un mapeo rank local–GPU.
- Descomponer halo, transferencia y comunicación.
- Distinguir MPI GPU-aware de staging por host.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

El rank local, no el global, suele determinar la GPU dentro de un nodo.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

Cuando hay más ranks que GPUs aparece compartición que debe ser intencional.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

GPU-aware MPI puede aceptar buffers de dispositivo, pero soporte, ruta y sincronización deben verificarse.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- afinidad — vínculo entre trabajo y recursos físicos
- sobresuscripción — más entidades ejecutables que recursos asignados
- perfil — atribución del tiempo a regiones o fases


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Topologia Hibrida

![Nodos con ranks, hilos y GPU local](../../images/topologia-hibrida.svg)

**Cómo leerlo.** Recorre la jerarquía de afuera hacia adentro. El mapeo correcto conserva afinidad local y evita asignar accidentalmente varios ranks al mismo dispositivo.

### Offload Host Device

![Flujo de datos entre host y dispositivo](../../images/offload-host-device.svg)

**Cómo leerlo.** Separa preparación, H2D, kernel, D2H y validación. Esa separación evita llamar tiempo total a una medición que solo cubre el kernel.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "07"
NOTEBOOK = "07_hibrido/02_mpi_gpu.ipynb"
assert (ROOT / "curso" / "notebooks" / "07_hibrido" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Asignación de dispositivos

**Situación.** Se detecta sobresuscripción a partir de ranks locales y GPUs visibles.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
local_ranks, gpus = 6, 4
assignment = {rank: rank % gpus for rank in range(local_ranks)}
users = {gpu: [rank for rank, selected in assignment.items() if selected == gpu] for gpu in range(gpus)}
print(assignment)
for gpu, ranks in users.items(): print("GPU", gpu, "ranks", ranks, "compartida", len(ranks)>1)
assert set(assignment.values()) == set(range(gpus))


### Explicación del resultado

Si la política exige un rank por GPU, la asignación debe fallar en lugar de compartir silenciosamente.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Volumen de halo

**Situación.** Se calcula comunicación por paso para una malla 3D descompuesta en una dimensión.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
ny, nz, layers, bytes_per_value = 512, 256, 2, 8
one_face = ny * nz * layers * bytes_per_value
interior_rank = 2 * one_face
print({"one_face_MiB": one_face/2**20, "interior_rank_MiB": interior_rank/2**20})
assert interior_rank == 2 * one_face


### Lectura razonada

El modelo se combina con ancho de banda PCIe/NVLink y red para decidir staging, empaquetado y solapamiento.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. ¿Qué falla funcional o de rendimiento puede ocurrir si dos ranks locales seleccionan la misma GPU sin intención?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Registrar rank global/local, bus id y modelo de GPU.
2. Comparar staging host y GPU-aware cuando ambos existan.
3. Medir interior, halo, red y sincronización con la misma entrada.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.


## Errores frecuentes

- Seleccionar GPU con rank global.
- Asumir GPU-aware por aceptar un puntero.
- Sincronizar todo el dispositivo y eliminar el solapamiento.


## Criterios de aceptación

- Mapeo proceso–GPU inequívoco.
- Halos validados contra referencia.
- Ruta de comunicación y sincronización documentadas.


## Síntesis

- La pregunta que debes poder responder es: **¿Cómo asignar procesos a dispositivos y solapar halos sin ocultar transferencias?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Entornos de clúster](../../../topicos_avanzados/ENTORNOS_CLUSTER.md)
- [Protocolo hardware](../../../docs/REPRODUCIBILIDAD_EJERCICIOS.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 07](README.md)
